In [3]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
import requests
import pandas as pd
from pyspark.sql import SparkSession

# ヨーロッパ主要都市の座標
cities = {
    "Amsterdam": {"lat": 52.3676, "lon": 4.9041, "country": "Netherlands"},
    "Paris": {"lat": 48.8566, "lon": 2.3522, "country": "France"},
    "London": {"lat": 51.5074, "lon": -0.1278, "country": "United Kingdom"},
    "Berlin": {"lat": 52.5200, "lon": 13.4050, "country": "Germany"},
    "Rome": {"lat": 41.9028, "lon": 12.4964, "country": "Italy"},
    "Barcelona": {"lat": 41.3851, "lon": 2.1734, "country": "Spain"},
    "Stockholm": {"lat": 59.3293, "lon": 18.0686, "country": "Sweden"},
    "Copenhagen": {"lat": 55.6761, "lon": 12.5683, "country": "Denmark"},
    "Vienna": {"lat": 48.2082, "lon": 16.3738, "country": "Austria"},
    "Lisbon": {"lat": 38.7169, "lon": -9.1399, "country": "Portugal"},
    "Pula": {"lat": 44.8666, "lon": 13.8496, "country": "Croatia"},
    "Budapest": {"lat": 47.4979, "lon": 19.0402, "country": "Hungary"}
}

print(f"都市数: {len(cities)}")
print(cities)

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 10, Finished, Available, Finished, False)

都市数: 12
{'Amsterdam': {'lat': 52.3676, 'lon': 4.9041, 'country': 'Netherlands'}, 'Paris': {'lat': 48.8566, 'lon': 2.3522, 'country': 'France'}, 'London': {'lat': 51.5074, 'lon': -0.1278, 'country': 'United Kingdom'}, 'Berlin': {'lat': 52.52, 'lon': 13.405, 'country': 'Germany'}, 'Rome': {'lat': 41.9028, 'lon': 12.4964, 'country': 'Italy'}, 'Barcelona': {'lat': 41.3851, 'lon': 2.1734, 'country': 'Spain'}, 'Stockholm': {'lat': 59.3293, 'lon': 18.0686, 'country': 'Sweden'}, 'Copenhagen': {'lat': 55.6761, 'lon': 12.5683, 'country': 'Denmark'}, 'Vienna': {'lat': 48.2082, 'lon': 16.3738, 'country': 'Austria'}, 'Lisbon': {'lat': 38.7169, 'lon': -9.1399, 'country': 'Portugal'}, 'Pula': {'lat': 44.8666, 'lon': 13.8496, 'country': 'Croatia'}, 'Budapest': {'lat': 47.4979, 'lon': 19.0402, 'country': 'Hungary'}}


In [3]:
import time

# 各都市の気温データを取得
all_temperature_data = []

for city_name, city_info in cities.items():
    print(f"{city_name}のデータを取得中...")
    
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": city_info["lat"],
        "longitude": city_info["lon"],
        "start_date": "2000-01-01",
        "end_date": "2023-12-31",
        "daily": "temperature_2m_mean",
        "timezone": "auto"
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    # 日次データを年次平均に変換
    dates = data["daily"]["time"]
    temps = data["daily"]["temperature_2m_mean"]
    
    df_temp = pd.DataFrame({"date": dates, "temperature": temps})
    df_temp["year"] = pd.to_datetime(df_temp["date"]).dt.year
    df_temp["city"] = city_name
    df_temp["country"] = city_info["country"]
    
    # 年次平均を計算
    df_annual = df_temp.groupby(["city", "country", "year"])["temperature"].mean().reset_index()
    df_annual.columns = ["city", "country", "year", "avg_temperature"]
    
    all_temperature_data.append(df_annual)
    time.sleep(1)  # APIへの負荷を減らすため

# 全都市のデータを結合
df_all_temps = pd.concat(all_temperature_data, ignore_index=True)
print(f"取得完了！総レコード数: {len(df_all_temps)}")
display(df_all_temps.head(20))

StatementMeta(, ef573d59-50a6-4498-90cb-cd8b19e50cad, 6, Finished, Cancelled, Cancelled, False)

Amsterdamのデータを取得中...
Parisのデータを取得中...
Londonのデータを取得中...
Berlinのデータを取得中...
Romeのデータを取得中...


In [4]:
import requests

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 41.9028,  # Rome
    "longitude": 12.4964,
    "start_date": "2000-01-01",
    "end_date": "2023-12-31",
    "daily": "temperature_2m_mean",
    "timezone": "auto"
}

response = requests.get(url, params=params)
print(f"Status: {response.status_code}")
print(response.json().keys())

StatementMeta(, ef573d59-50a6-4498-90cb-cd8b19e50cad, 7, Finished, Cancelled, Cancelled, False)

In [5]:
import requests
import pandas as pd
import time

all_temperature_data = []

for city_name, city_info in cities.items():
    print(f"{city_name}のデータを取得中...")
    
    url = "https://archive-api.open-meteo.com/v1/archive"
    # テスト用：2015〜2023年だけ（短い範囲）
    params = {
    "latitude": city_info["lat"],
    "longitude": city_info["lon"],
    "start_date": "2015-01-01",  # 短くする
    "end_date": "2023-12-31",
    "daily": "temperature_2m_mean",
    "timezone": "auto"
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    dates = data["daily"]["time"]
    temps = data["daily"]["temperature_2m_mean"]
    
    df_temp = pd.DataFrame({"date": dates, "temperature": temps})
    df_temp["year"] = pd.to_datetime(df_temp["date"]).dt.year
    df_temp["city"] = city_name
    df_temp["country"] = city_info["country"]
    
    df_annual = df_temp.groupby(["city", "country", "year"])["temperature"].mean().reset_index()
    df_annual.columns = ["city", "country", "year", "avg_temperature"]
    
    all_temperature_data.append(df_annual)
    print(f"{city_name}完了!")  # 都市ごとに完了メッセージ追加

df_all_temps = pd.concat(all_temperature_data, ignore_index=True)
print(f"全都市取得完了！総レコード数: {len(df_all_temps)}")
display(df_all_temps.head(20))

StatementMeta(, ef573d59-50a6-4498-90cb-cd8b19e50cad, 8, Submitted, Running, Running, True)

Amsterdamのデータを取得中...


In [2]:
import requests
import pandas as pd
import time

start_time = time.time()

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 52.3676,  # Amsterdam
    "longitude": 4.9041,
    "start_date": "2015-01-01",
    "end_date": "2023-12-31",
    "daily": "temperature_2m_mean",
    "timezone": "auto"
}

response = requests.get(url, params=params)
data = response.json()

end_time = time.time()
print(f"APIレスポンス時間: {end_time - start_time:.2f}秒")
print(f"データ件数: {len(data['daily']['time'])}")

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 9, Finished, Available, Finished, False)

APIレスポンス時間: 4.05秒
データ件数: 3287


In [21]:
import requests
import pandas as pd
import time

all_temperature_data = []

for city_name, city_info in cities_3.items():
    print(f"{city_name}のデータを取得中...")
    
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": city_info["lat"],
        "longitude": city_info["lon"],
        "start_date": "2000-01-01",
        "end_date": "2023-12-31",
        "daily": "temperature_2m_mean",
        "timezone": "auto"
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    dates = data["daily"]["time"]
    temps = data["daily"]["temperature_2m_mean"]
    
    df_temp = pd.DataFrame({"date": dates, "temperature": temps})
    df_temp["year"] = pd.to_datetime(df_temp["date"]).dt.year
    df_temp["city"] = city_name
    df_temp["country"] = city_info["country"]
    
    df_annual = df_temp.groupby(["city", "country", "year"])["temperature"].mean().reset_index()
    df_annual.columns = ["city", "country", "year", "avg_temperature"]
    
    all_temperature_data.append(df_annual)
    print(f"{city_name}完了!")

df_all_temps = pd.concat(all_temperature_data, ignore_index=True)
print(f"全都市取得完了！総レコード数: {len(df_all_temps)}")
display(df_all_temps.head(20))

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 43, Finished, Available, Finished, False)

Lisbonのデータを取得中...
Lisbon完了!
Pulaのデータを取得中...
Pula完了!
Budapestのデータを取得中...
Budapest完了!
全都市取得完了！総レコード数: 72


SynapseWidget(Synapse.DataFrame, d94c070c-4003-4eac-9e2f-be4d84107496)

In [7]:
cities_1 = {
    "Amsterdam": {"lat": 52.3676, "lon": 4.9041, "country": "Netherlands"},
    "Paris": {"lat": 48.8566, "lon": 2.3522, "country": "France"},
    "London": {"lat": 51.5074, "lon": -0.1278, "country": "United Kingdom"},
    "Berlin": {"lat": 52.5200, "lon": 13.4050, "country": "Germany"},
    "Rome": {"lat": 41.9028, "lon": 12.4964, "country": "Italy"},
}
print("グループ1定義完了!")

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 18, Finished, Available, Finished, False)

グループ1定義完了!


In [10]:
# グループ1の結果を一時保存
df_temps_group1 = df_all_temps.copy()
print(f"グループ1保存完了！レコード数: {len(df_temps_group1)}")
display(df_temps_group1)

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 23, Finished, Available, Finished, False)

グループ1保存完了！レコード数: 120


SynapseWidget(Synapse.DataFrame, b8c52c3c-45e2-45ff-94c4-35820d904c42)

In [15]:
cities_2 = {
    "Barcelona": {"lat": 41.3851, "lon": 2.1734, "country": "Spain"},
    "Stockholm": {"lat": 59.3293, "lon": 18.0686, "country": "Sweden"},
    "Copenhagen": {"lat": 55.6761, "lon": 12.5683, "country": "Denmark"},
    "Vienna": {"lat": 48.2082, "lon": 16.3738, "country": "Austria"}
}
print("グループ2定義完了!")

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 32, Finished, Available, Finished, False)

グループ2定義完了!


In [17]:
df_temps_group2 = df_all_temps.copy()
print(f"グループ2保存完了！レコード数: {len(df_temps_group2)}")

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 35, Finished, Available, Finished, False)

グループ2保存完了！レコード数: 96


In [18]:
cities_3 = {
    "Lisbon": {"lat": 38.7169, "lon": -9.1399, "country": "Portugal"},
    "Pula": {"lat": 44.8666, "lon": 13.8496, "country": "Croatia"},
    "Budapest": {"lat": 47.4979, "lon": 19.0402, "country": "Hungary"}
}
print("グループ3定義完了!")

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 36, Finished, Available, Finished, False)

グループ3定義完了!


In [22]:
# グループ3保存
df_temps_group3 = df_all_temps.copy()
print(f"グループ3保存完了！レコード数: {len(df_temps_group3)}")

# 全グループを結合
df_all_cities = pd.concat([df_temps_group1, df_temps_group2, df_temps_group3], ignore_index=True)
print(f"全都市結合完了！総レコード数: {len(df_all_cities)}")
display(df_all_cities)

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 45, Finished, Available, Finished, False)

グループ3保存完了！レコード数: 72
全都市結合完了！総レコード数: 288


SynapseWidget(Synapse.DataFrame, 0c72ecbb-84ca-4ae3-b51b-0e1a51d9d8ad)

In [23]:
# SparkのDataFrameに変換してDeltaテーブルとして保存
from pyspark.sql import SparkSession

df_spark = spark.createDataFrame(df_all_cities)

df_spark.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("bronze_temperature")

print("Bronzeテーブル保存完了!")

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 46, Finished, Available, Finished, False)

Bronzeテーブル保存完了!


In [24]:
# 観光・経済データを読み込む
df_tourism = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("Files/bronze/world_tourism_economy_data.csv")

display(df_tourism)

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 47, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 252ddc11-a5fa-47e4-8af6-006f5aadb8d8)

In [25]:
df_tourism.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("bronze_tourism")

print("観光・経済データ保存完了!")

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 48, Finished, Available, Finished, False)

観光・経済データ保存完了!


In [26]:
# ヨーロッパの国コードリスト
european_countries = ['NLD', 'FRA', 'GBR', 'DEU', 'ITA', 'ESP', 
                       'SWE', 'DNK', 'AUT', 'PRT', 'HRV', 'HUN']

# ヨーロッパのみフィルタ・必要な列だけ選択
from pyspark.sql.functions import col

df_tourism_silver = df_tourism.filter(
    col("country_code").isin(european_countries)
).select(
    "country",
    "country_code",
    "year",
    "tourism_arrivals",
    "tourism_receipts",
    "gdp",
    "inflation"
)

display(df_tourism_silver)

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 49, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 956f223a-d9c3-4383-9704-5b6e9f102799)

In [27]:
df_tourism_silver.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("silver_tourism")

print("Silver観光データ保存完了!")

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 50, Finished, Available, Finished, False)

Silver観光データ保存完了!


In [28]:
# 気温データの国名を確認
spark.table("bronze_temperature").select("city", "country").distinct().show()

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 51, Finished, Available, Finished, False)

+----------+--------------+
|      city|       country|
+----------+--------------+
| Stockholm|        Sweden|
|Copenhagen|       Denmark|
|     Paris|        France|
|    London|United Kingdom|
| Amsterdam|   Netherlands|
|    Vienna|       Austria|
|      Pula|       Croatia|
|  Budapest|       Hungary|
|    Lisbon|      Portugal|
| Barcelona|         Spain|
|      Rome|         Italy|
|    Berlin|       Germany|
+----------+--------------+



In [29]:
from pyspark.sql.functions import col

# 国名と国コードのマッピング
country_mapping = {
    "Netherlands": "NLD",
    "France": "FRA",
    "United Kingdom": "GBR",
    "Germany": "DEU",
    "Italy": "ITA",
    "Spain": "ESP",
    "Sweden": "SWE",
    "Denmark": "DNK",
    "Austria": "AUT",
    "Portugal": "PRT",
    "Croatia": "HRV",
    "Hungary": "HUN"
}

# マッピングを使って気温データに国コードを追加
from pyspark.sql.functions import create_map, lit
from itertools import chain

mapping_expr = create_map([lit(x) for x in chain(*country_mapping.items())])

df_temp_with_code = spark.table("bronze_temperature")\
    .withColumn("country_code", mapping_expr[col("country")])

display(df_temp_with_code.head(5))

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 53, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b9a27381-a31e-4736-882f-db22b444f8da)

In [30]:
# 観光データと気温データをcountry_codeと年でJOIN
df_gold = df_tourism_silver.join(
    df_temp_with_code,
    (df_tourism_silver["country_code"] == df_temp_with_code["country_code"]) &
    (df_tourism_silver["year"] == df_temp_with_code["year"]),
    "left"
).select(
    df_tourism_silver["country"],
    df_tourism_silver["country_code"],
    df_tourism_silver["year"],
    df_tourism_silver["tourism_arrivals"],
    df_tourism_silver["tourism_receipts"],
    df_tourism_silver["gdp"],
    df_tourism_silver["inflation"],
    df_temp_with_code["city"],
    df_temp_with_code["avg_temperature"]
)

display(df_gold)

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 55, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d4236cac-1c4d-4515-a643-0374ef930132)

In [31]:
from pyspark.sql.functions import col

# 2000年以降でフィルタしてGoldテーブルを保存
df_gold_filtered = df_gold.filter(col("year") >= 2000)

df_gold_filtered.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("gold_european_analysis")

print(f"Goldテーブル保存完了！レコード数: {len(df_gold_filtered.collect())}")

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 56, Finished, Available, Finished, False)

Goldテーブル保存完了！レコード数: 288


In [33]:
%%sql
SELECT country, city, year, tourism_arrivals, gdp, avg_temperature,inflation
FROM gold_european_analysis
WHERE year = 2020
ORDER BY country

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 58, Finished, Available, Finished, False)

<Spark SQL result set with 12 rows and 7 fields>

In [34]:
%%sql
SELECT country, year, inflation, tourism_arrivals, avg_temperature
FROM gold_european_analysis
ORDER BY inflation DESC
LIMIT 10

StatementMeta(, fa9e8fe1-7896-49a7-996a-33ca4c633fda, 59, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 5 fields>

In [1]:
%%sql
SELECT year, COUNT(*) 
FROM gold_european_analysis
GROUP BY year
ORDER BY year

StatementMeta(, 764efa86-1636-4a41-8ccd-730be01e23fe, 2, Finished, Available, Finished, False)

<Spark SQL result set with 24 rows and 2 fields>

In [2]:
%%sql
SELECT year, COUNT(*), SUM(tourism_arrivals) 
FROM gold_european_analysis
GROUP BY year
ORDER BY year

StatementMeta(, 764efa86-1636-4a41-8ccd-730be01e23fe, 3, Finished, Available, Finished, False)

<Spark SQL result set with 24 rows and 3 fields>

In [3]:
%%sql
SELECT year, country, tourism_arrivals 
FROM bronze_tourism
WHERE year >= 2020
AND country_code = 'FRA'
ORDER BY year

StatementMeta(, 764efa86-1636-4a41-8ccd-730be01e23fe, 4, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 3 fields>